# Módulo 9 · Clase 1 — Obtención de Datos

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## Contexto del proyecto

A lo largo de este módulo construiremos un sistema completo de extremo a extremo:

```
Datos → Limpieza → SQL → ML → DL → API REST → Documentación → Despliegue
```

**Dataset hilo conductor:** Wine Quality (UCI Machine Learning Repository)
- 1,599 vinos tintos con 11 características físico-químicas
- Objetivo: predecir la calidad (score 0-10) y clasificar si es "buen vino" (≥7)
- URL pública descargable → ideal para demostrar la fase de obtención real

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Fuentes de datos: tipos y estrategias |
| 2 | Descarga con `requests` |
| 3 | Leer CSV, JSON y APIs REST públicas |
| 4 | Validación inicial del dataset |
| 5 | Guardar datos crudos de forma reproducible |

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import hashlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Crear estructura de directorios del proyecto
for directorio in ['data/raw', 'data/processed', 'data/db',
                   'models', 'api', 'logs', 'docs']:
    Path(directorio).mkdir(parents=True, exist_ok=True)

print("Estructura del proyecto creada:")
for d in Path('.').iterdir():
    if d.is_dir() and not d.name.startswith('.'):
        print(f"  {d}/")

---
## 1. Fuentes de datos: tipos y estrategias

```
Tipos de fuente:                    Herramienta principal:
  Archivo CSV/Excel/JSON            → pandas.read_*
  URL pública (CSV, JSON)           → requests + pandas
  API REST                          → requests
  Base de datos SQL                 → sqlite3, psycopg2, SQLAlchemy
  Web scraping                      → BeautifulSoup, Scrapy, Playwright
  Streaming / Kafka                 → kafka-python
  Cloud (S3, GCS, Azure Blob)       → boto3, google-cloud, azure-storage

Principios de la obtención profesional:
  1. Reproducible → misma semilla, misma URL, mismo resultado
  2. Trazable     → registrar fuente, fecha, hash del archivo
  3. Cruda        → guardar los datos SIN modificar (raw)
  4. Verificada   → comprobar integridad (hash MD5/SHA256)
  5. Documentada  → metadatos junto al dato
```

---
## 2. Descarga con `requests`

In [ ]:
# URL pública del dataset de vinos (UCI ML Repository)
URL_VINO_TINTO  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
URL_VINO_BLANCO = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

def descargar_dataset(url, ruta_local, forzar=False):
    """
    Descarga un archivo desde una URL si no existe localmente.
    Devuelve metadatos de la descarga.
    """
    ruta = Path(ruta_local)

    if ruta.exists() and not forzar:
        print(f"  Ya existe: {ruta} → usando caché")
    else:
        print(f"  Descargando: {url}")
        respuesta = requests.get(url, timeout=30)
        respuesta.raise_for_status()  # lanza excepción si falla (4xx, 5xx)
        ruta.write_bytes(respuesta.content)
        print(f"  Guardado en: {ruta}")

    # Calcular hash para verificación de integridad
    contenido = ruta.read_bytes()
    sha256    = hashlib.sha256(contenido).hexdigest()
    tamaño    = ruta.stat().st_size

    return {
        'url':       url,
        'ruta':      str(ruta),
        'tamaño_kb': round(tamaño / 1024, 2),
        'sha256':    sha256[:16] + '...', # primeros 16 chars
        'fecha':     datetime.now().isoformat()
    }

print("Descargando datasets de vino:")
meta_tinto  = descargar_dataset(URL_VINO_TINTO,  'data/raw/vino_tinto_raw.csv')
meta_blanco = descargar_dataset(URL_VINO_BLANCO, 'data/raw/vino_blanco_raw.csv')

print(f"\nMetadatos vino tinto:  {meta_tinto}")
print(f"Metadatos vino blanco: {meta_blanco}")

# Guardar metadatos de descarga
metadatos_descarga = {'tinto': meta_tinto, 'blanco': meta_blanco}
with open('data/raw/descarga_metadatos.json', 'w') as f:
    json.dump(metadatos_descarga, f, indent=2)
print("\nMetadatos guardados en data/raw/descarga_metadatos.json")

In [ ]:
# Leer el CSV descargado
# El separador es ';' (no ',') — importante inspeccionarlo antes de asumir
print("Primeras líneas del archivo crudo:")
with open('data/raw/vino_tinto_raw.csv', 'r') as f:
    for i, linea in enumerate(f):
        print(f"  {linea.rstrip()}")
        if i >= 3:
            break

print("\n→ Separador: ';'  (no la coma por defecto)")

# Cargar correctamente
df_tinto  = pd.read_csv('data/raw/vino_tinto_raw.csv',  sep=';')
df_blanco = pd.read_csv('data/raw/vino_blanco_raw.csv', sep=';')

# Añadir columna tipo
df_tinto['tipo']  = 'tinto'
df_blanco['tipo'] = 'blanco'

# Combinar
df = pd.concat([df_tinto, df_blanco], ignore_index=True)

print(f"\nDataFrame combinado: {df.shape}")
print(f"  Tintos:  {len(df_tinto):,}")
print(f"  Blancos: {len(df_blanco):,}")
print(f"  Total:   {len(df):,}")
df.head(3)

---
## 3. Leer JSON y APIs REST públicas

In [ ]:
# Demo: consumir una API REST pública
# Usamos la API de Open Notify (posición ISS) — gratuita y sin autenticación

def obtener_dato_api(url, timeout=10):
    """Hace GET a una API REST y devuelve el JSON como dict."""
    respuesta = requests.get(url, timeout=timeout)
    respuesta.raise_for_status()
    return respuesta.json()

# Astronautas en el espacio ahora mismo
try:
    datos_iss = obtener_dato_api('http://api.open-notify.org/astros.json')
    print("API Open Notify — Astronautas en el espacio:")
    print(json.dumps(datos_iss, indent=2))
    df_astros = pd.DataFrame(datos_iss['people'])
    print(f"\nDataFrame: {df_astros.shape}")
    print(df_astros)
except Exception as e:
    print(f"API no disponible ({e}) — demo con datos simulados:")
    datos_iss = {
        'people': [
            {'name': 'Oleg Kononenko', 'craft': 'ISS'},
            {'name': 'Nikolai Chub',   'craft': 'ISS'},
        ],
        'number': 2
    }
    print(json.dumps(datos_iss, indent=2))

In [ ]:
# Patrones comunes de lectura de datos
print("Patrones de lectura con pandas:")
print("""
  # CSV con coma
  df = pd.read_csv('archivo.csv')

  # CSV con punto y coma
  df = pd.read_csv('archivo.csv', sep=';')

  # CSV desde URL directamente
  df = pd.read_csv('https://example.com/datos.csv')

  # Excel
  df = pd.read_excel('archivo.xlsx', sheet_name='Hoja1')

  # JSON (lista de objetos)
  df = pd.read_json('archivo.json')

  # JSON desde API (dict → DataFrame manual)
  datos = requests.get(url).json()
  df = pd.DataFrame(datos['items'])

  # Parquet (binario comprimido, muy eficiente)
  df = pd.read_parquet('archivo.parquet')

  # SQL
  import sqlite3
  conn = sqlite3.connect('base.db')
  df = pd.read_sql('SELECT * FROM tabla', conn)
""")

---
## 4. Validación inicial del dataset

In [ ]:
def informe_inicial(df, nombre='Dataset'):
    """Genera un informe rápido de un DataFrame recién cargado."""
    print(f"\n{'='*55}")
    print(f"  INFORME INICIAL — {nombre}")
    print(f"{'='*55}")
    print(f"  Filas:    {df.shape[0]:,}")
    print(f"  Columnas: {df.shape[1]}")
    print(f"  Memoria:  {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

    print(f"\n  Tipos de datos:")
    for col, dtype in df.dtypes.items():
        nulls = df[col].isna().sum()
        print(f"    {col:30} {str(dtype):10} nulls={nulls}")

    print(f"\n  Duplicados exactos: {df.duplicated().sum()}")

    num_cols = df.select_dtypes(include='number').columns.tolist()
    if num_cols:
        print(f"\n  Estadísticas numéricas (primeras 5 cols):")
        print(df[num_cols[:5]].describe().round(3).to_string())

informe_inicial(df, 'Vinos combinados (tinto + blanco)')

In [ ]:
# Inspección visual rápida
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
cols_num = df.select_dtypes(include='number').columns.tolist()

for ax, col in zip(axes.ravel(), cols_num):
    df_t = df[df['tipo']=='tinto'][col]
    df_b = df[df['tipo']=='blanco'][col]
    ax.hist(df_t, bins=30, alpha=0.6, color='firebrick',  label='Tinto',  density=True)
    ax.hist(df_b, bins=30, alpha=0.6, color='steelblue', label='Blanco', density=True)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel(''); ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Ocultar el subplot extra
axes.ravel()[-1].set_visible(False)

plt.suptitle('Distribuciones por variable y tipo de vino', fontsize=14)
plt.tight_layout()
plt.show()

# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de calidad (total)'); axes[0].set_xlabel('Puntuación')
axes[0].grid(True, alpha=0.3, axis='y')

for tipo, color in [('tinto','firebrick'), ('blanco','steelblue')]:
    df[df['tipo']==tipo]['quality'].value_counts().sort_index().plot(
        kind='line', ax=axes[1], color=color, lw=2, marker='o', label=tipo)
axes[1].set_title('Calidad por tipo'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Distribución de calidad:")
print(df['quality'].value_counts().sort_index().to_string())

---
## 5. Guardar datos crudos de forma reproducible

In [ ]:
# Guardar el dataset combinado SIN modificar
# Usar Parquet: más eficiente que CSV para datos tabulares
ruta_raw = 'data/raw/vinos_combinados_raw.parquet'
df.to_parquet(ruta_raw, index=False)
print(f"Dataset crudo guardado: {ruta_raw}")

# También en CSV como backup legible
df.to_csv('data/raw/vinos_combinados_raw.csv', index=False)
print("Backup CSV: data/raw/vinos_combinados_raw.csv")

# Guardar metadatos del dataset
meta_dataset = {
    'nombre':        'Wine Quality (UCI)',
    'fuente':        'https://archive.ics.uci.edu/ml/datasets/wine+quality',
    'citacion':      'Cortez et al., 2009. Wine Quality.',
    'fecha_descarga': datetime.now().isoformat(),
    'filas':         int(df.shape[0]),
    'columnas':      int(df.shape[1]),
    'columnas_lista': df.columns.tolist(),
    'variable_objetivo': 'quality',
    'tipos': {'tinto': int(df['tipo'].eq('tinto').sum()),
              'blanco': int(df['tipo'].eq('blanco').sum())},
    'calidad_min': int(df['quality'].min()),
    'calidad_max': int(df['quality'].max()),
    'sha256_raw': hashlib.sha256(Path(ruta_raw).read_bytes()).hexdigest()[:16]
}

with open('data/raw/dataset_metadatos.json', 'w', encoding='utf-8') as f:
    json.dump(meta_dataset, f, indent=2, ensure_ascii=False)

print(f"\nMetadatos del dataset:")
print(json.dumps(meta_dataset, indent=2))

In [ ]:
# Verificar que los datos se pueden leer correctamente
df_verificado = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
assert df_verificado.shape == df.shape, "Error: los shapes no coinciden"
assert list(df_verificado.columns) == list(df.columns), "Error: columnas distintas"
print(f"✓ Verificación correcta: {df_verificado.shape}")
print(f"  Columnas: {list(df_verificado.columns)}")

---
## Práctica

### Ejercicio 1 — Obtener datos desde otra API pública

**Enunciado:** La API de Open Meteo (https://open-meteo.com) es gratuita y devuelve datos meteorológicos.

```
URL: https://api.open-meteo.com/v1/forecast
Parámetros: latitude=43.35, longitude=-8.41, hourly=temperature_2m&forecast_days=3
(coordenadas de A Coruña)
```

1. Haz GET a la URL con `requests.get(url, params=dict)`
2. Convierte el JSON a DataFrame
3. Muestra las primeras filas y plotea la temperatura por hora

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import requests
import pandas as pd
import matplotlib.pyplot as plt

url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude':  43.35,
    'longitude': -8.41,
    'hourly':    'temperature_2m',
    'forecast_days': 3
}

try:
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    datos = resp.json()

    df_met = pd.DataFrame({
        'tiempo':      pd.to_datetime(datos['hourly']['time']),
        'temperatura': datos['hourly']['temperature_2m']
    })

    print(f"Datos obtenidos: {df_met.shape}")
    print(df_met.head())

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_met['tiempo'], df_met['temperatura'], 'steelblue', lw=2)
    ax.fill_between(df_met['tiempo'], df_met['temperatura'], alpha=0.2)
    ax.set_title('Temperatura prevista en A Coruña (próximas 72h)')
    ax.set_xlabel('Hora'); ax.set_ylabel('°C')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"Error: {e}")
```
</details>

---
### Ejercicio 2 — Manejo robusto de errores HTTP

**Enunciado:** En producción los errores de red son inevitables. Mejora la función `descargar_dataset` para:

1. Reintentar hasta 3 veces si falla (con espera entre intentos)
2. Registrar cada intento en un log (`logs/descargas.log`)
3. Devolver `None` si agota los reintentos (sin lanzar excepción)

Pista: usa `time.sleep()` y un bucle `for`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import requests
import time
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(
    filename='logs/descargas.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s'
)

def descargar_con_reintentos(url, ruta_local, max_reintentos=3, espera=2):
    ruta = Path(ruta_local)
    for intento in range(1, max_reintentos + 1):
        try:
            logging.info(f"Intento {intento}/{max_reintentos}: {url}")
            print(f"Intento {intento}/{max_reintentos}: {url}")
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            ruta.parent.mkdir(parents=True, exist_ok=True)
            ruta.write_bytes(resp.content)
            logging.info(f"OK: {ruta} ({len(resp.content)/1024:.1f} KB)")
            print(f"  ✓ Descargado: {len(resp.content)/1024:.1f} KB")
            return ruta
        except requests.RequestException as e:
            logging.warning(f"Fallo intento {intento}: {e}")
            print(f"  ✗ Error: {e}")
            if intento < max_reintentos:
                print(f"  Esperando {espera}s antes del siguiente intento...")
                time.sleep(espera)
    logging.error(f"Todos los intentos fallaron: {url}")
    print(f"✗ No se pudo descargar tras {max_reintentos} intentos")
    return None

# Probar
resultado = descargar_con_reintentos(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
    'data/raw/test_reintentos.csv'
)
print(f"\nResultado: {resultado}")
print("Log guardado en logs/descargas.log")
```
</details>

---
### Ejercicio 3 — Validador de esquema

**Enunciado:** Crea una función `validar_esquema(df, esquema_esperado)` donde `esquema_esperado` es un dict `{columna: dtype_esperado}`. La función debe:

1. Verificar que todas las columnas esperadas existen
2. Verificar que los tipos coinciden (al menos la familia: `float`, `int`, `object`)
3. Verificar que no hay valores nulos
4. Devolver un dict `{'ok': bool, 'errores': list[str]}`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import pandas as pd
import numpy as np

def validar_esquema(df, esquema_esperado):
    errores = []

    for col, tipo_esperado in esquema_esperado.items():
        # 1. Columna existe
        if col not in df.columns:
            errores.append(f"Columna faltante: '{col}'")
            continue

        # 2. Tipo de dato
        dtype_real = df[col].dtype
        if tipo_esperado == 'float' and not pd.api.types.is_float_dtype(dtype_real):
            errores.append(f"'{col}': esperado float, encontrado {dtype_real}")
        elif tipo_esperado == 'int' and not pd.api.types.is_integer_dtype(dtype_real):
            errores.append(f"'{col}': esperado int, encontrado {dtype_real}")
        elif tipo_esperado == 'object' and not pd.api.types.is_object_dtype(dtype_real):
            errores.append(f"'{col}': esperado object, encontrado {dtype_real}")

        # 3. Sin nulos
        n_nulos = df[col].isna().sum()
        if n_nulos > 0:
            errores.append(f"'{col}': {n_nulos} valores nulos")

    return {'ok': len(errores) == 0, 'errores': errores}

# Probar con el dataset de vinos
df = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')

esquema = {
    'fixed acidity':       'float',
    'volatile acidity':    'float',
    'citric acid':         'float',
    'residual sugar':      'float',
    'chlorides':           'float',
    'free sulfur dioxide': 'float',
    'total sulfur dioxide':'float',
    'density':             'float',
    'pH':                  'float',
    'sulphates':           'float',
    'alcohol':             'float',
    'quality':             'int',
    'tipo':                'object',
}

resultado = validar_esquema(df, esquema)
print(f"Validación: {'✓ OK' if resultado['ok'] else '✗ FALLÓ'}")
if resultado['errores']:
    for e in resultado['errores']:
        print(f"  - {e}")
else:
    print("  Todos los tipos y nulos verificados correctamente")
```
</details>

---
### Ejercicio 4 — Descarga incremental

**Enunciado:** En sistemas de producción los datos llegan en lotes. Simula una descarga incremental:

1. Divide el dataset en 3 "lotes" (batch_001, batch_002, batch_003)
2. Guarda cada lote como `data/raw/batch_NNN.parquet`
3. Crea una función `cargar_todos_los_lotes(directorio)` que los combine
4. Verifica que el resultado es igual al dataset original

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')

# Dividir en 3 lotes
n = len(df)
lotes = np.array_split(df, 3)

for i, lote in enumerate(lotes, 1):
    ruta = f'data/raw/batch_{i:03d}.parquet'
    lote.to_parquet(ruta, index=False)
    print(f"Lote {i:03d}: {len(lote):,} filas → {ruta}")

def cargar_todos_los_lotes(directorio):
    rutas = sorted(Path(directorio).glob('batch_*.parquet'))
    if not rutas:
        raise FileNotFoundError(f"No hay lotes en {directorio}")
    lotes = [pd.read_parquet(r) for r in rutas]
    print(f"Cargados {len(lotes)} lotes: {[len(l) for l in lotes]}")
    return pd.concat(lotes, ignore_index=True)

df_reconstruido = cargar_todos_los_lotes('data/raw')
print(f"\nDataset reconstruido: {df_reconstruido.shape}")

assert df_reconstruido.shape == df.shape, "Shapes distintos"
assert (df_reconstruido.values == df.values).all(), "Contenido distinto"
print("✓ Verificación OK: dataset reconstruido idéntico al original")
```
</details>

---
### Ejercicio 5 (reto) — Pipeline de ingesta completa

**Enunciado:** Crea una clase `IngestaPipeline` que encapsule todo el proceso:

```python
pipeline = IngestaPipeline(
    fuentes={'tinto': URL_TINTO, 'blanco': URL_BLANCO},
    directorio_raw='data/raw',
    esquema_esperado={'quality': 'int', 'alcohol': 'float'}
)
df = pipeline.ejecutar()  # descarga + combina + valida + guarda + devuelve
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import requests, json, hashlib, logging
import pandas as pd
from pathlib import Path
from datetime import datetime

class IngestaPipeline:
    def __init__(self, fuentes, directorio_raw, esquema_esperado=None):
        self.fuentes    = fuentes  # dict {nombre: url}
        self.dir_raw    = Path(directorio_raw)
        self.esquema    = esquema_esperado or {}
        self.dir_raw.mkdir(parents=True, exist_ok=True)
        self.log        = []

    def _descargar(self, nombre, url):
        ruta = self.dir_raw / f'{nombre}_raw.csv'
        if not ruta.exists():
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            ruta.write_bytes(r.content)
        self.log.append({'accion': 'descarga', 'nombre': nombre, 'ruta': str(ruta)})
        return ruta

    def _cargar(self, nombre, ruta):
        df = pd.read_csv(ruta, sep=';')
        df['tipo'] = nombre
        return df

    def _validar(self, df):
        errores = []
        for col, tipo in self.esquema.items():
            if col not in df.columns:
                errores.append(f"Falta columna: {col}")
            elif df[col].isna().any():
                errores.append(f"Nulos en: {col}")
        return errores

    def ejecutar(self):
        dataframes = []
        for nombre, url in self.fuentes.items():
            ruta = self._descargar(nombre, url)
            df   = self._cargar(nombre, ruta)
            dataframes.append(df)
            print(f"  {nombre}: {df.shape}")

        df_final = pd.concat(dataframes, ignore_index=True)

        errores = self._validar(df_final)
        if errores:
            raise ValueError(f"Validación fallida: {errores}")

        ruta_out = self.dir_raw / 'ingesta_final.parquet'
        df_final.to_parquet(ruta_out, index=False)

        meta = {'filas': len(df_final), 'fecha': datetime.now().isoformat(), 'log': self.log}
        (self.dir_raw / 'ingesta_meta.json').write_text(json.dumps(meta, indent=2))

        print(f"\nIngesta completada: {df_final.shape} → {ruta_out}")
        return df_final


pipeline = IngestaPipeline(
    fuentes={
        'tinto':  'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
        'blanco': 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv',
    },
    directorio_raw='data/raw',
    esquema_esperado={'quality': 'int', 'alcohol': 'float'}
)
df = pipeline.ejecutar()
print(df.head())
```
</details>

---
## Resumen

```python
import requests
import pandas as pd
from pathlib import Path

# Descargar archivo
resp = requests.get(url, timeout=30)
resp.raise_for_status()
Path('data/raw/archivo.csv').write_bytes(resp.content)

# Leer CSV con separador no estándar
df = pd.read_csv('archivo.csv', sep=';')

# Consumir API REST
datos = requests.get(url, params={'key': 'value'}).json()
df = pd.DataFrame(datos['items'])

# Guardar datos crudos (Parquet = eficiente)
df.to_parquet('data/raw/dataset_raw.parquet', index=False)

# Verificar integridad
import hashlib
sha = hashlib.sha256(Path('archivo.parquet').read_bytes()).hexdigest()

# Principios:
# 1. Guardar datos CRUDOS sin modificar
# 2. Registrar fuente + fecha + hash
# 3. Manejar errores de red (reintentos)
# 4. Validar esquema al cargar
```

## ¿Qué viene en la próxima clase?
En la **Clase 2** limpiaremos y prepararemos los datos: tratamiento de outliers, encoding, feature engineering y guardado del dataset listo para modelar.